# RAG pipeline for Zenith Bank

This notebook contains a RAG system for a ficticious digial bank.

## Load Text from PDF files

In [1]:
import pymupdf

from pathlib import Path

In [2]:
notebook_location = Path.cwd()
project_root = notebook_location if (notebook_location / "data").exists() else notebook_location.parent
data_dir = project_root / "data" / "raw"

# Get all PDFs
pdf_files = sorted(data_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs")

Found 20 PDFs


In [3]:
documents = []
for pdf_path in pdf_files:
    doc = pymupdf.open(pdf_path)
    doc_id = pdf_path.stem  # Use the filename without extension as the document ID
    num_pages = len(doc)  # Get page count BEFORE closing
    
    text = ""
    for page in doc:
        text += page.get_text()
    
    doc.close()
    
    documents.append({
        "id": doc_id,
        "filename": pdf_path.name,
        "path": str(pdf_path),
        "text": text,
        "num_pages": num_pages,
        "num_chars": len(text)
    })
    print(f"✓ {pdf_path.name}: {num_pages} pages, {len(text):,} chars")

# Quick inspection
print(f"\nTotal documents: {len(documents)}")
print(f"Sample text from first doc:\n{documents[0]['text'][:200]}...")

✓ 01_linea_credito_inicial_y_aumentos.pdf: 2 pages, 3,928 chars
✓ 02_comisiones_y_costos.pdf: 1 pages, 1,815 chars
✓ 03_msi_y_tarjeta_virtual.pdf: 2 pages, 2,474 chars
✓ 04_seguridad_y_proteccion_antifraude.pdf: 2 pages, 3,149 chars
✓ 05_formas_de_pago_y_automatizacion.pdf: 2 pages, 2,741 chars
✓ 06_reporte_a_sociedades_de_informacion_crediticia.pdf: 1 pages, 1,853 chars
✓ 07_historial_crediticio_y_atrasos.pdf: 2 pages, 1,952 chars
✓ 08_cancelacion_por_fallecimiento.pdf: 1 pages, 1,493 chars
✓ 09_cuenta_digital_y_ahorros.pdf: 1 pages, 1,795 chars
✓ 10_pago_automatico_paso_a_paso.pdf: 2 pages, 2,018 chars
✓ 11_introduccion_tarjeta_garantizada.pdf: 2 pages, 2,931 chars
✓ 12_dinero_reservado_uso_retiro_rendimiento.pdf: 2 pages, 2,520 chars
✓ 13_comparacion_garantizada_regular_prepago.pdf: 2 pages, 1,920 chars
✓ 14_requisitos_y_contratacion_garantizada.pdf: 2 pages, 2,343 chars
✓ 15_sistema_de_misiones_introduccion.pdf: 2 pages, 2,714 chars
✓ 16_primeras_misiones_paso_a_paso.pdf: 2 pages, 

In [4]:
documents[0]

{'id': '01_linea_credito_inicial_y_aumentos',
 'filename': '01_linea_credito_inicial_y_aumentos.pdf',
 'path': '/Users/lilianabadillo/Projects/repo/rag-zenith-bank/data/raw/01_linea_credito_inicial_y_aumentos.pdf',
 'text': "Guía Completa: Línea de Crédito Inicial y Aumentos —\nZenith Digital Bank\nEste documento explica en detalle cómo Zenith Digital Bank determina la línea de crédito inicial de\nun cliente, qué factores influyen en aumentos posteriores y qué comportamientos favorecen un\ncrecimiento constante del límite disponible. Está dirigido a clientes que desean entender el\nrazonamiento detrás de las decisiones de crédito y a equipos internos que necesiten explicar estas\npolíticas de forma consistente.\n1. Cómo se calcula tu línea de crédito inicial\nCuando un cliente recibe su tarjeta por primera vez, Zenith Digital Bank asigna una línea de crédito\ninicial con base en un análisis integral de su perfil financiero. Este análisis no depende únicamente\ndel ingreso reportado, si

## Split by Token Chunks

In [5]:
chunk_size = 500
overlap = 50

def chunk_text(text, chunk_size=1000, overlap=200):
    """Split text into chunks with overlap."""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
    return chunks

for index, doc in enumerate(documents):
    chunks = chunk_text(doc['text'], chunk_size=chunk_size, overlap=overlap)
    doc['chunks'] = chunks
    doc['num_chunks'] = len(chunks)
    print(f"{doc['filename']}: {len(doc['chunks'])} chunks")
    

01_linea_credito_inicial_y_aumentos.pdf: 9 chunks
02_comisiones_y_costos.pdf: 5 chunks
03_msi_y_tarjeta_virtual.pdf: 6 chunks
04_seguridad_y_proteccion_antifraude.pdf: 7 chunks
05_formas_de_pago_y_automatizacion.pdf: 7 chunks
06_reporte_a_sociedades_de_informacion_crediticia.pdf: 5 chunks
07_historial_crediticio_y_atrasos.pdf: 5 chunks
08_cancelacion_por_fallecimiento.pdf: 4 chunks
09_cuenta_digital_y_ahorros.pdf: 4 chunks
10_pago_automatico_paso_a_paso.pdf: 5 chunks
11_introduccion_tarjeta_garantizada.pdf: 7 chunks
12_dinero_reservado_uso_retiro_rendimiento.pdf: 6 chunks
13_comparacion_garantizada_regular_prepago.pdf: 5 chunks
14_requisitos_y_contratacion_garantizada.pdf: 6 chunks
15_sistema_de_misiones_introduccion.pdf: 7 chunks
16_primeras_misiones_paso_a_paso.pdf: 5 chunks
17_mision_de_acumulacion_y_consolidacion.pdf: 5 chunks
18_que_pasa_si_no_completo_una_mision.pdf: 4 chunks
19_cuando_y_como_se_otorgan_aumentos.pdf: 6 chunks
20_pago_de_la_tarjeta_garantizada_y_manejo_de_incumpli

In [6]:
documents[0]

{'id': '01_linea_credito_inicial_y_aumentos',
 'filename': '01_linea_credito_inicial_y_aumentos.pdf',
 'path': '/Users/lilianabadillo/Projects/repo/rag-zenith-bank/data/raw/01_linea_credito_inicial_y_aumentos.pdf',
 'text': "Guía Completa: Línea de Crédito Inicial y Aumentos —\nZenith Digital Bank\nEste documento explica en detalle cómo Zenith Digital Bank determina la línea de crédito inicial de\nun cliente, qué factores influyen en aumentos posteriores y qué comportamientos favorecen un\ncrecimiento constante del límite disponible. Está dirigido a clientes que desean entender el\nrazonamiento detrás de las decisiones de crédito y a equipos internos que necesiten explicar estas\npolíticas de forma consistente.\n1. Cómo se calcula tu línea de crédito inicial\nCuando un cliente recibe su tarjeta por primera vez, Zenith Digital Bank asigna una línea de crédito\ninicial con base en un análisis integral de su perfil financiero. Este análisis no depende únicamente\ndel ingreso reportado, si

## Create embeddings with sentence_transformers

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")

In [8]:
from sentence_transformers import SentenceTransformer

/Users/lilianabadillo/Projects/repo/rag-zenith-bank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
model_name = 'jaimevera1107/all-MiniLM-L6-v2-similarity-es'
model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 24058.21it/s]


In [10]:
for doc in documents:
    doc['embeddings'] = [model.encode(chunk) for chunk in doc['chunks']]
    print(f"{doc['filename']}: {len(doc['embeddings'])} embeddings")

01_linea_credito_inicial_y_aumentos.pdf: 9 embeddings
02_comisiones_y_costos.pdf: 5 embeddings
03_msi_y_tarjeta_virtual.pdf: 6 embeddings
04_seguridad_y_proteccion_antifraude.pdf: 7 embeddings
05_formas_de_pago_y_automatizacion.pdf: 7 embeddings
06_reporte_a_sociedades_de_informacion_crediticia.pdf: 5 embeddings
07_historial_crediticio_y_atrasos.pdf: 5 embeddings
08_cancelacion_por_fallecimiento.pdf: 4 embeddings
09_cuenta_digital_y_ahorros.pdf: 4 embeddings
10_pago_automatico_paso_a_paso.pdf: 5 embeddings
11_introduccion_tarjeta_garantizada.pdf: 7 embeddings
12_dinero_reservado_uso_retiro_rendimiento.pdf: 6 embeddings
13_comparacion_garantizada_regular_prepago.pdf: 5 embeddings
14_requisitos_y_contratacion_garantizada.pdf: 6 embeddings
15_sistema_de_misiones_introduccion.pdf: 7 embeddings
16_primeras_misiones_paso_a_paso.pdf: 5 embeddings
17_mision_de_acumulacion_y_consolidacion.pdf: 5 embeddings
18_que_pasa_si_no_completo_una_mision.pdf: 4 embeddings
19_cuando_y_como_se_otorgan_aumen

In [11]:
print(f"Number of chunks for first document: {documents[0]['num_chunks']}")

print(f"Number of embeddings for first document: {len(documents[0]['embeddings'])}")

Number of chunks for first document: 9
Number of embeddings for first document: 9


## Store the embeddings in ChromaDB

In [12]:
import chromadb

from chromadb.utils import embedding_functions

In [13]:
sentence_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=model_name
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 26321.41it/s]


In [16]:
chromadb_dir = project_root / "chroma_db"

client_vectordb = chromadb.PersistentClient(path=str(chromadb_dir))

In [17]:
collection = client_vectordb.get_or_create_collection(name="zenith_bank_docs_sentence_transformer")

In [18]:
documents[0]

{'id': '01_linea_credito_inicial_y_aumentos',
 'filename': '01_linea_credito_inicial_y_aumentos.pdf',
 'path': '/Users/lilianabadillo/Projects/repo/rag-zenith-bank/data/raw/01_linea_credito_inicial_y_aumentos.pdf',
 'text': "Guía Completa: Línea de Crédito Inicial y Aumentos —\nZenith Digital Bank\nEste documento explica en detalle cómo Zenith Digital Bank determina la línea de crédito inicial de\nun cliente, qué factores influyen en aumentos posteriores y qué comportamientos favorecen un\ncrecimiento constante del límite disponible. Está dirigido a clientes que desean entender el\nrazonamiento detrás de las decisiones de crédito y a equipos internos que necesiten explicar estas\npolíticas de forma consistente.\n1. Cómo se calcula tu línea de crédito inicial\nCuando un cliente recibe su tarjeta por primera vez, Zenith Digital Bank asigna una línea de crédito\ninicial con base en un análisis integral de su perfil financiero. Este análisis no depende únicamente\ndel ingreso reportado, si

In [19]:
for doc in documents:
    print(f"{doc['filename']}: {len(doc['embeddings'])} embeddings")
    ids = [f"{doc['id']}_chunk_{i}" for i in range(len(doc['embeddings']))]
    embs = [embedding for embedding in doc['embeddings']]
    text = [chunk for chunk in doc['chunks']]
    metadata = [{"id": doc['id'], "chunk_index": i, "total_chunks": len(doc['embeddings'])} for i in range(len(doc['embeddings']))]
    
    collection.add(
        ids=ids,
        embeddings=embs,
        metadatas=metadata,
        documents=text
    )

01_linea_credito_inicial_y_aumentos.pdf: 9 embeddings
02_comisiones_y_costos.pdf: 5 embeddings
03_msi_y_tarjeta_virtual.pdf: 6 embeddings
04_seguridad_y_proteccion_antifraude.pdf: 7 embeddings
05_formas_de_pago_y_automatizacion.pdf: 7 embeddings
06_reporte_a_sociedades_de_informacion_crediticia.pdf: 5 embeddings
07_historial_crediticio_y_atrasos.pdf: 5 embeddings
08_cancelacion_por_fallecimiento.pdf: 4 embeddings
09_cuenta_digital_y_ahorros.pdf: 4 embeddings
10_pago_automatico_paso_a_paso.pdf: 5 embeddings
11_introduccion_tarjeta_garantizada.pdf: 7 embeddings
12_dinero_reservado_uso_retiro_rendimiento.pdf: 6 embeddings
13_comparacion_garantizada_regular_prepago.pdf: 5 embeddings
14_requisitos_y_contratacion_garantizada.pdf: 6 embeddings
15_sistema_de_misiones_introduccion.pdf: 7 embeddings
16_primeras_misiones_paso_a_paso.pdf: 5 embeddings
17_mision_de_acumulacion_y_consolidacion.pdf: 5 embeddings
18_que_pasa_si_no_completo_una_mision.pdf: 4 embeddings
19_cuando_y_como_se_otorgan_aumen

In [27]:
query = "¿Cuáles son las formas de pago disponibles en Zenith Bank?"
embedding_query = model.encode(query)
top_k = 5

results = collection.query(
    query_embeddings=[embedding_query],
    n_results=top_k
)

for result in results['documents'][0]:
    print(result)
    print("-----" * 5)

Formas de Pago y Automatización — Zenith Digital
Bank
Este documento detalla los distintos canales disponibles para pagar la Tarjeta de Crédito Zenith
Digital Bank, así como el funcionamiento de la función de pago automático, actualmente en fase de
expansión.
1. Canales de pago disponibles
Los clientes pueden pagar su tarjeta de crédito directamente desde la aplicación, tocando el botón
'Pagar' en la pantalla principal. A partir de ahí, existen distintos canales que se ajustan a las
preferencias
-------------------------
Primeras Misiones Paso a Paso — Zenith Digital Bank
Este documento describe en detalle las dos primeras misiones del sistema de construcción de límite
de la Tarjeta Garantizada Zenith Digital Bank, incluyendo lo que el cliente debe hacer para
completarlas correctamente.
1. Primera misión: activar el dinero reservado
La primera misión del ciclo consiste en transferir, desde la Cuenta Digital hacia la Tarjeta
Garantizada, cualquier monto que el cliente desee utilizar com

In [28]:
query = "¿Puedo pagar a meses sin intereses?"
embedding_query = model.encode(query)
top_k = 5

results = collection.query(
    query_embeddings=[embedding_query],
    n_results=top_k
)

for result in results['documents'][0]:
    print(result)
    print("-----" * 5)

ción, en la sección correspondiente a Meses Sin Intereses, donde se
actualiza de forma periódica conforme se incorporan nuevos aliados comerciales.
G
Los planes a MSI se pueden liquidar de forma anticipada, total o parcial.
G
Adelantar mensualidades genera un descuento proporcional sobre el saldo restante.
G
La disponibilidad de MSI depende del comercio y de la promoción vigente en cada momento.
2. Tarjeta virtual: una capa adicional de seguridad
Todo cliente con Tarjeta de Crédito recibe automá
-------------------------
, visible solo en la app
Característica
Tarjeta física
Tarjeta virtual
Uso recomendado
Compras físicas y cajeros
Compras y pagos en línea
¿Se puede regenerar?
No sin reposición completa
Sí, en cualquier momento
desde la app

-------------------------
ente dentro del plazo, esto puede disminuir las
probabilidades de recibir un aumento de límite de crédito. Sin embargo, mientras las misiones sigan
disponibles dentro del periodo vigente, el cliente puede intentarlo nuevam

In [29]:
query = "¿Cómo puedo obtener un aumento en mi límite de crédito?"
embedding_query = model.encode(query)
top_k = 5

results = collection.query(
    query_embeddings=[embedding_query],
    n_results=top_k
)

for result in results['documents'][0]:
    print(result)
    print("-----" * 5)

liente actual como a futuras solicitudes de crédito.

-------------------------
ífica para
quienes no cuentan aún con historial crediticio suficiente: la posibilidad de acceder a un producto de
crédito real definiendo ellos mismos el monto de su línea, en lugar de depender completamente de
una evaluación de riesgo tradicional que podría resultar en un rechazo o en una línea muy limitada.
G
El cliente decide su propio límite de crédito, hasta $20,000 MXN.
G
Incluye prácticamente los mismos beneficios que la tarjeta regular (MSI, sin anualidad, red
Mastercard).
G
El buen uso 
-------------------------
SI, sin anualidad, red
Mastercard).
G
El buen uso puede derivar en aumentos de crédito adicionales al dinero reservado.

-------------------------
ente dentro del plazo, esto puede disminuir las
probabilidades de recibir un aumento de límite de crédito. Sin embargo, mientras las misiones sigan
disponibles dentro del periodo vigente, el cliente puede intentarlo nuevamente.
Elemento
Detalle
P